## Polytope time series

In this notebook you will see how to:

- use Polytope to retrieve a timeseries at a specific location of interest
- aggregate the data over the ensemble and create a simple timeseries plot
- create a meteogram

### Components of earthkit

This tutorial uses the following earthkit components - click any logo to open the package documentation:

<div align="center">
  <br>
  <a href="https://earthkit-data.readthedocs.io/en/latest/" target="_blank" style="display:inline-block; margin: 0 15px;">
    <img src="https://raw.githubusercontent.com/ecmwf/logos/refs/heads/main/logos/earthkit/earthkit-data-light.svg" alt="earthkit-data" width="200">
  </a>
  <a href="https://earthkit-transforms.readthedocs.io/en/latest/" target="_blank" style="display:inline-block; margin: 0 15px;">
    <img src="https://raw.githubusercontent.com/ecmwf/logos/refs/heads/main/logos/earthkit/earthkit-transforms-light.svg" alt="earthkit-transforms" width="200">
  </a>
  <a href="https://earthkit-plots.readthedocs.io/en/latest/" target="_blank" style="display:inline-block; margin: 0 15px;">
    <img src="https://raw.githubusercontent.com/ecmwf/logos/refs/heads/main/logos/earthkit/earthkit-plots-light.svg" alt="earthkit-plots" width="200">
  </a>
</div>

### 1. Data retrieval

First, retrieve ENS time series data using Polytope.

In [ ]:
import earthkit.data as ekd

# lat, lon
location = [38.78655345978706, -9.109280931080349]

request = {
    "class": "od",
    "stream" : "enfo",
    "type" : "pf",
    "date" : -1,
    "time" : "0000",
    "levtype" : "sfc",
    "expver" : 1, 
    "domain" : "g",
    "param" : "164/167/169",
    "number" : "1/to/50",
    "step": "0/to/360",
    "feature" : {
        "type" : "timeseries",
        "points": [location],
        "axes": "step",
    },
}

ds = ekd.from_source("polytope", "ecmwf-mars", request, stream=False, address='polytope.ecmwf.int')
ds.to_xarray()

### 2. Aggregate and plot a simple timeseries

We can compute simple aggregations easily using earthkit-transforms.

In [ ]:
import earthkit.transforms as ekt
import earthkit.plots as ekp

mean_xr_ds = ekt.ensemble.mean(ds.to_xarray())

ekp.timeseries.line(
    mean_xr_ds['2t'],
    x="t",
    x_units="celsius",
    title="ERA5 hourly {variable_name} at {latitude:%Lt} {longitude:%Ln}"
).show()

### 3. Plot a meteogram directly

More powerful plots are available out of the box with earthkit-plots. We can directly plot the retrieved data as an ENS meteogram.

In [ ]:
# TODO: fix issue in ekp
from earthkit.plots.interactive import Chart

TIME_FREQUENCY = "6h"
QUANTILES = [0, 0.1, 0.25, 0.5, 0.75, 0.9, 1]

chart = Chart()
chart.title("ECMWF ensemble meteogram at {latitude:%Lt} {longitude:%Ln}")
chart.box(ds, time_frequency=TIME_FREQUENCY, quantiles=QUANTILES)
chart.line(ds, aggregation='mean', line_color='grey', time_frequency=TIME_FREQUENCY)
chart.show(renderer="png")  # Replace with chart.show() in an interactive session!